# Spectral Clustering and the Fiedler Vector

Chapter 7: Symmetric Matrix · *Mathematics for AI and Machine Learning* by Fuheng Wu

A live 3D demo: each node of two linked clusters is lifted to its entry in the Fiedler vector; the groups settle at two levels and the edges crossing zero are the cut, next to all eight Laplacian eigenvalues.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-7/spectral-clustering/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's default weights: two groups of four nodes (0-3 and 4-7)
bridge = 0.3        # the weak edge 3-4 between the groups
cross = 0.0         # an extra edge 1-6 between the groups
inner = 1.0         # the edge 0-1 inside the first group
normalized = True   # normalized Laplacian (True) or L = D - W (False)

W = torch.zeros(8, 8, dtype=torch.float64)
for group in ([0, 1, 2, 3], [4, 5, 6, 7]):
    for a in group:
        for b in group:
            if a < b:
                W[a, b] = W[b, a] = 1.0
W[0, 1] = W[1, 0] = inner
W[3, 4] = W[4, 3] = bridge
W[1, 6] = W[6, 1] = cross

# Normalized Laplacian  L = I - D^{-1/2} W D^{-1/2}  (symmetric), or L = D - W
d = W.sum(dim=1)
D_inv_sqrt = torch.diag(d.rsqrt())
if normalized:
    L = torch.eye(8, dtype=torch.float64) - D_inv_sqrt @ W @ D_inv_sqrt
else:
    L = torch.diag(d) - W

# Spectral theorem: real eigenvalues, orthonormal eigenvectors (ascending)
lam, Q = torch.linalg.eigh(L)
y = Q[:, 1]
fiedler = D_inv_sqrt @ y if normalized else y   # embedding D^{-1/2} y for the normalized Laplacian
fiedler = fiedler * (-1 if fiedler[0] > 0 else 1)   # sign fixed so the first group reads negative

# Rayleigh quotient of that eigenvector equals lambda_1
print("eigenvalues:", [round(v, 3) for v in lam.tolist()])
print("lambda_1 =", round(lam[1].item(), 4), " Rayleigh quotient =", round((y @ L @ y / (y @ y)).item(), 4))
print("Fiedler vector:", [round(v, 3) for v in fiedler.tolist()])
print("cluster A:", [i for i in range(8) if fiedler[i] < 0])
print("cluster B:", [i for i in range(8) if fiedler[i] >= 0])

eigenvalues: [-0.0, 0.044, 1.242, 1.333, 1.333, 1.333, 1.333, 1.38]
lambda_1 = 0.0439  Rayleigh quotient = 0.0439
Fiedler vector: [-0.209, -0.209, -0.209, -0.181, 0.181, 0.209, 0.209, 0.209]
cluster A: [0, 1, 2, 3]
cluster B: [4, 5, 6, 7]
